# Gold V2 — Silver V2 → reproducible ML-ready splits

Chỉ nhận **Silver V2 report** đã được publish, xác minh SHA-256 và cleaning audit. Không ghi đè Gold V1. Mặc định dừng nếu còn review/guardrail. Notebook không training model và không đánh giá test.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/Ko4rut/vietnamese-multimodal-moderation.git"
BRANCH = "feat/gold-v2-cleaned-dataset"
PROJECT_DIR = Path("/content/vmm_gold_v2")
if (PROJECT_DIR / ".git").is_dir():
    dirty = subprocess.run(["git", "-C", str(PROJECT_DIR), "status", "--porcelain"],
                           capture_output=True, text=True, check=True)
    if dirty.stdout.strip():
        raise RuntimeError("Local repo has uncommitted changes")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
else:
    if PROJECT_DIR.exists() and any(PROJECT_DIR.iterdir()):
        raise RuntimeError(f"Non-git project directory: {PROJECT_DIR}")
    subprocess.run(["git", "clone", "--single-branch", "--branch", BRANCH,
                    REPO_URL, str(PROJECT_DIR)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."],
               cwd=str(PROJECT_DIR), check=True)

# Ensure the current Colab kernel resolves the src-layout package.
SRC_DIR = (PROJECT_DIR / "src").resolve()
if not (SRC_DIR / "vietnamese_topic_classification" / "__init__.py").is_file():
    raise FileNotFoundError(f"Missing Python package under {SRC_DIR}")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
from vietnamese_topic_classification.data.gold import GoldConfig, GoldReader
from vietnamese_topic_classification.data.gold.pipeline_v2 import GoldV2Pipeline
from vietnamese_topic_classification.data.silver import SilverReader
print("Gold V2 imported from", SRC_DIR)


In [ ]:
from google.colab import drive

MOUNT = "/content/vtsnlp_drive"
drive.mount(MOUNT)
DATA_LAKE_ROOT = Path(MOUNT) / "MyDrive" / "vietnamese-multilabel-topic-classification"
reports = sorted((DATA_LAKE_ROOT / "05_reports").glob("silver_quality_v2_*.json"))
if not reports:
    raise FileNotFoundError(
        "No published Silver V2 quality report. Run Silver V2 notebook with PUBLISH=True first."
    )
if len(reports) != 1:
    raise RuntimeError(
        "Multiple Silver V2 quality reports: select a report explicitly after checking audit: "
        + ", ".join(x.name for x in reports)
    )
SILVER_V2_REPORT_NAME = reports[0].name
print("Selected Silver V2:", SILVER_V2_REPORT_NAME)


In [ ]:
import json

silver = SilverReader(DATA_LAKE_ROOT, SILVER_V2_REPORT_NAME)
verified_silver = silver.verify()
print("Silver V2 verification:", verified_silver)
print("Silver V2 cleaning:", {key: silver.report.get(key) for key in
      ("cleaned_rows", "review_rows", "guardrail_rows", "rejected_rows")})

# Treat review/guardrail rows as unresolved until their audit is accepted.
ALLOW_UNREVIEWED = False
gold = GoldV2Pipeline(silver, config=GoldConfig(seed=42),
                      allow_unreviewed=ALLOW_UNREVIEWED)
# Verify source audit and fail closed if review rows remain.
source_report, source_verification = gold._source()
OUTPUT_FILES = gold.planned_paths(source_verification["sha256"])
for key, path in OUTPUT_FILES.items():
    print(key, path)
if any(path.with_name(path.name + ".uploading").exists() for path in OUTPUT_FILES.values()):
    raise FileExistsError("Stale Gold V2 .uploading artifact; inspect before rerunning")
existing = [path for path in OUTPUT_FILES.values() if path.exists()]
if existing and len(existing) != len(OUTPUT_FILES):
    raise FileExistsError("Partial Gold V2 publication; inspect before rerunning")
GOLD_ALREADY_EXISTS = bool(existing)


In [ ]:
import os

if GOLD_ALREADY_EXISTS:
    reader = GoldReader(DATA_LAKE_ROOT, OUTPUT_FILES["manifest"].name)
    print("Gold V2 already exists, verified:", reader.verify())
    assert reader.manifest["source"]["silver_sha256"] == verified_silver["sha256"]
else:
    env = os.environ.copy()
    env["PYTHONPATH"] = os.pathsep.join(
        x for x in (str(SRC_DIR), env.get("PYTHONPATH", "")) if x
    )
    command = [
        sys.executable, str(PROJECT_DIR / "scripts/10_build_gold_v2.py"),
        "--data-lake-root", str(DATA_LAKE_ROOT),
        "--report-name", SILVER_V2_REPORT_NAME,
        "--seed", "42",
    ]
    if ALLOW_UNREVIEWED:
        command.append("--allow-unreviewed")
    subprocess.run(command, cwd=str(PROJECT_DIR), env=env, check=True)
    print("Gold V2 published.")


In [ ]:
from collections import Counter

reader = GoldReader(DATA_LAKE_ROOT, OUTPUT_FILES["manifest"].name)
verification = reader.verify()
assert reader.manifest["pipeline_version"] == "v2"
assert reader.manifest["source"]["silver_sha256"] == verified_silver["sha256"]
assert verification["records"] == verified_silver["records"]
seen = set()
for split in ("train", "validation", "test"):
    counts = Counter()
    for row in reader.iter_split(split):
        assert row["id"] not in seen, "Cross-split duplicate ID"
        assert row["label_id"] == reader.manifest["label_to_id"][row["domain"]]
        assert row["text_length"] == len(row["text"])
        seen.add(row["id"])
        counts[row["domain"]] += 1
    print(split, verification["split_rows"][split], "rows,", len(counts), "labels")
assert len(seen) == verification["records"]
print("GOLD V2 VERIFIED:", OUTPUT_FILES["manifest"])
